In [1]:
from pathlib import Path

import pandas as pd

FILES = {
    "mortgage": "data/raw/mortgage_rates.csv",
    "listing": "data/processed/combined_listings_pruned.csv",
    "sold": "data/processed/combined_sold_pruned.csv",
}

# scripts/preprocessing/mortgage -> repo root
base_dir = Path.cwd().parents[2]

sold = pd.read_csv(base_dir / FILES["sold"], low_memory=False)
listing = pd.read_csv(base_dir / FILES["listing"], low_memory=False)
mortgage = pd.read_csv(base_dir / FILES["mortgage"])

Convert to `datetime64`

In [2]:
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
listing['year_month'] = pd.to_datetime(listing['ListingContractDate']).dt.to_period('M')
mortgage['year_month'] = pd.to_datetime(mortgage['year_month']).dt.to_period('M')

Merging Lists

In [3]:
sold_with_rates = sold.merge(mortgage, on='year_month', how='left')
listings_with_rates = listing.merge(mortgage, on='year_month', how='left')

Sanity Check for rates

In [4]:
print(sold_with_rates['rate'].isnull().sum())
print(listings_with_rates['rate'].isnull().sum())

0
0


In [5]:
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate']].head())
print(f"sold: {len(sold_with_rates)}")
print(f"listings: {len(listings_with_rates)}")

    CloseDate year_month  ClosePrice    rate
0  2024-04-16    2024-04   1360000.0  6.9925
1  2024-04-30    2024-04    413700.0  6.9925
2  2024-04-02    2024-04    250000.0  6.9925
3  2024-04-01    2024-04    526000.0  6.9925
4  2024-04-29    2024-04   1525000.0  6.9925
sold: 458336
listings: 547162


In [ ]:
import os

sold_with_rates.to_csv(base_dir / "data/processed/sold_with_rates.csv", index=False)
listings_with_rates.to_csv(base_dir / "data/processed/listings_with_rates.csv", index=False)
os.remove(base_dir / FILES["listing"])
os.remove(base_dir / FILES["sold"])
